In [1]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage
from langchain.tools import tool
from langchain.agents import create_agent
import base64

load_dotenv()

c:\Users\Sumit Sah\tutorial-agentic-ai\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Sumit Sah\tutorial-agentic-ai\.venv\Lib\site-packages\langgraph\checkpoint\serde\encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


True

In [3]:
with open("blood_work.png", "rb") as f:
    image_b64 = base64.b64encode(f.read()).decode()

print(image_b64[:100])
print(len(image_b64))

iVBORw0KGgoAAAANSUhEUgAAAmwAAAHgCAIAAACXbaZMAACxv0lEQVR4nOzdeVwT1944/iGBkASIIKuyBBAQIahYqiBuiCjWrVi8
60748


In [4]:
llm = ChatGroq(model="qwen/qwen3.8-27b")

message = HumanMessage(content=[
    {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}"}},
    {"type": "text",      "text": "This is a blood work report. Extract all test results and flag any values outside the normal range."}
])

response = llm.invoke([message])
print(response.content)

Based on the blood work report provided for **Rajesh Sharma (Age 48, Male)** dated **May 7, 2026**, here is a summary of the test results.

Values that are **outside the normal range** are flagged with ⚠️.

### **COMPLETE BLOOD COUNT (CBC)**
*   **Hemoglobin:** 15.1 g/dL (Normal: 13.5 – 17.5)
*   **Hematocrit:** 44% (Normal: 41 – 53%)
*   **WBC:** 6.8 x 10^3/uL (Normal: 4.5 – 11.0)
*   **Platelets:** 220 x 10^3/uL (Normal: 150 – 400)

### **LIPID PANEL**
*   **Total Cholesterol:** 238 mg/dL ⚠️ **(High)**
    *   *Normal Range: < 200*
*   **LDL Cholesterol:** 162 mg/dL ⚠️ **(High)**
    *   *Normal Range: < 100*
*   **HDL Cholesterol:** 36 mg/dL ⚠️ **(Low)**
    *   *Normal Range: > 40*
*   **Triglycerides:** 188 mg/dL ⚠️ **(High)**
    *   *Normal Range: < 150*

### **METABOLIC PANEL**
*   **Glucose (Fasting):** 92 mg/dL (Normal: 70 – 99)
*   **HbA1c:** 5.3% (Normal: < 5.7%)
*   **Creatinine:** 1.0 mg/dL (Normal: 0.7 – 1.3)
*   **eGFR:** 82 mL/min (Normal: > 60)

***

### **Summary of 

In [5]:
DIET_GUIDE = {
    "high cholesterol": {
        "avoid": ["Ghee, butter and vanaspati", "Deep-fried snacks (samosa, pakora, puri)", "Full-fat paneer and cream", "Red meat and organ meat"],
        "eat_more": ["Oats and barley", "Rajma, chana and moong dal", "Almonds and walnuts (a small handful)", "Flaxseed (alsi)"],
    },
    "high ldl": {
        "avoid": ["Ghee, butter and vanaspati", "Bakery items (biscuits, cakes, khari)", "Processed meats"],
        "eat_more": ["Oats and psyllium husk (isabgol)", "Soluble-fibre vegetables (bhindi, brinjal)", "Mustard or rice bran oil in small amounts"],
    },
    "low hdl": {
        "avoid": ["Refined sugar and sweets", "Trans fats in packaged snacks"],
        "eat_more": ["Walnuts, almonds and flaxseed", "Fatty fish (rohu, mackerel) 2x a week", "Daily brisk walking (30 min) alongside diet"],
    },
    "high triglycerides": {
        "avoid": ["Sugar, jaggery and sweets", "White rice and maida in large portions", "Sugary drinks and fruit juices", "Alcohol"],
        "eat_more": ["Whole grains (jowar, bajra, ragi)", "Fatty fish or flaxseed for omega-3", "Non-starchy vegetables"],
    },
    "high glucose": {
        "avoid": ["Sweets, sugar and sugary drinks", "White bread, maida and polished rice", "Potatoes in large portions"],
        "eat_more": ["Millets (ragi, jowar, bajra)", "Methi seeds and bitter gourd (karela)", "Dal and sprouts for protein"],
    },
    "low hemoglobin": {
        "avoid": ["Tea or coffee right after meals (blocks iron absorption)"],
        "eat_more": ["Spinach, methi and other leafy greens", "Jaggery, dates and pomegranate", "Vitamin C foods (amla, lemon) with meals"],
    },
    "low vitamin d": {
        "avoid": ["Staying indoors all day"],
        "eat_more": ["Morning sunlight 15-20 min", "Fortified milk and curd", "Egg yolks and fatty fish"],
    },
    "low vitamin b12": {
        "avoid": ["Skipping dairy entirely if vegetarian"],
        "eat_more": ["Milk, curd and paneer", "Eggs", "Fortified cereals"],
    },
}


In [8]:
@tool
def get_diet_recommendations(abnormal_findings: list[str]) -> str:
    """Get Indian diet recommendations for abnormal blood test findings.

    Args:
        abnormal_findings: Abnormal results from the blood report, using these names:
            "high cholesterol", "high ldl", "low hdl", "high triglycerides", "high glucose",
            "low hemoglobin", "low vitamin d", "low vitamin b12".
    """
    sections, unknown = [], []
    for finding in abnormal_findings:
        guide = DIET_GUIDE.get(finding.strip().lower())
        if guide is None:
            unknown.append(finding)
            continue
        sections.append(
            f"{finding.upper()}\n"
            f"  Avoid: {'; '.join(guide['avoid'])}\n"
            f"  Eat more: {'; '.join(guide['eat_more'])}"
        )
    if unknown:
        sections.append(f"No diet guidance available for: {', '.join(unknown)}")
    return "\n\n".join(sections) if sections else "No abnormal findings provided."


print(get_diet_recommendations.invoke({"abnormal_findings": ["high ldl", "low hdl", "high triglycerides"]}))

diet_agent = create_agent(
    llm,
    tools=[get_diet_recommendations],
    system_prompt=(
        "You are a health assistant. When given a blood report image: "
        "1) extract every test result and flag values outside the reference range as HIGH or LOW, "
        "2) call get_diet_recommendations with the abnormal findings (e.g. 'high ldl', 'low hdl'), "
        "3) reply with a short summary of the results followed by the diet plan from the tool. "
        "Do not diagnose or prescribe medication, and remind the user to consult a doctor."
    ),
)

report_message = HumanMessage(content=[
    {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}"}},
    {"type": "text", "text": "Analyze my blood report and give me diet recommendations."},
])

result = diet_agent.invoke({"messages": [report_message]})

for msg in result["messages"]:
    if getattr(msg, "tool_calls", None):
        print("Tool called:", msg.tool_calls[0]["name"], msg.tool_calls[0]["args"])

print(result["messages"][-1].text)

HIGH LDL
  Avoid: Ghee, butter and vanaspati; Bakery items (biscuits, cakes, khari); Processed meats
  Eat more: Oats and psyllium husk (isabgol); Soluble-fibre vegetables (bhindi, brinjal); Mustard or rice bran oil in small amounts

LOW HDL
  Avoid: Refined sugar and sweets; Trans fats in packaged snacks
  Eat more: Walnuts, almonds and flaxseed; Fatty fish (rohu, mackerel) 2x a week; Daily brisk walking (30 min) alongside diet

HIGH TRIGLYCERIDES
  Avoid: Sugar, jaggery and sweets; White rice and maida in large portions; Sugary drinks and fruit juices; Alcohol
  Eat more: Whole grains (jowar, bajra, ragi); Fatty fish or flaxseed for omega-3; Non-starchy vegetables
Tool called: get_diet_recommendations {'abnormal_findings': ['high cholesterol', 'high ldl', 'low hdl', 'high triglycerides']}
## Summary of Your Blood Report (Rajesh Sharma, age 48)

**Normal results:** All CBC values (hemoglobin, hematocrit, WBC, platelets), fasting glucose, HbA1c, creatinine, and eGFR are within range.

